# 승부예측용 데이터셋 생성

## Imports

In [1]:
import pandas as pd
import json
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import os

# matplotlib 한글 폰트 설정
if os.name == 'nt':
    plt.rc('font', family='Malgun Gothic')
elif os.name == 'posix':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

## 챔피언 태그 확인

In [2]:
# read chamnpion_tag
champ_tag = pd.read_csv('champ_tag.csv', index_col=0)
champ_tag.drop(columns=['name'], inplace=True)
champ_tag.fillna(0, inplace=True)
champ_tag = champ_tag.astype(int)

In [3]:
tag_lineup_dict = {}
for tag_name in champ_tag.columns.to_list():
    tag_lineup_dict['me_'+tag_name] = []
    tag_lineup_dict['en_'+tag_name] = []
tag_lineup_dict['win'] = []

## 데이터 추출 및 적재

In [ ]:
# read match data

match_list = pd.read_csv('../data/match_id_list.csv')['match_id'].to_list()

for match_id in tqdm(match_list):
    with open(f'../data/match/{match_id}.json', 'r') as f:
        match_data = json.load(f)
    # 다시하기 제외
    if match_data['info']['gameDuration'] < 300:
        continue
    
    for tag_name in champ_tag.columns.to_list():
        tag_lineup_dict['me_'+tag_name].append(0)
        tag_lineup_dict['en_'+tag_name].append(0)
    
    # BLUE 팀기준
    for participant in match_data['info']['participants']:
        if participant['teamId'] == 100:
            team = 'me'
        else:
            team = 'en'
        
        for tag_name in champ_tag.columns.to_list():
            if participant['championId'] in champ_tag[champ_tag[tag_name] == 1].index:
                tag_lineup_dict[f'{team}_{tag_name}'][-1] += 1
    tag_lineup_dict['win'].append(match_data['info']['participants'][0]['win'])
    # RED 팀기준
    for participant in match_data['info']['participants']:
        if participant['teamId'] == 200:
            team = 'me'
        else:
            team = 'en'
        
        for tag_name in champ_tag.columns.to_list():
            if participant['championId'] in champ_tag[champ_tag[tag_name] == 1].index:
                tag_lineup_dict[f'{team}_{tag_name}'][-1] += 1
    tag_lineup_dict['win'].append(not match_data['info']['participants'][0]['win'])

# dataframe으로 변환
tag_lineup_df = pd.DataFrame(tag_lineup_dict)
# 확인
tag_lineup_df.head()

  0%|          | 0/60208 [00:00<?, ?it/s]

## 저장